# 프롬프트별 정확도 분석 및 시각화

이 노트북에서는 다음을 수행합니다:
1. **기본 정확도 분석**: 프롬프트별, 모델별, 문항유형별 정확도
2. **심화 분석**: 프롬프트 효과, 난이도 분석, 일치도 분석
3. **시각화**: 히트맵, 그룹 막대 그래프, 선 그래프


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# matplotlib 폰트 매니저
import matplotlib.font_manager as fm

# 스타일 설정 (먼저)
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

# 한글 폰트 설정 (스타일 설정 후에 해야 덮어쓰이지 않음)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

# 설정 확인
print(f"현재 폰트: {plt.rcParams['font.family']}")
print("한글 폰트 설정 완료!")


In [ ]:
# 데이터 경로 설정
data_path = Path('data')

# 정렬 순서 정의
MODEL_ORDER = ['gpt-5', 'gpt-4o', 'gpt-4o-mini', 'med42', 'medichat']
QTYPE_ORDER = ['MCQ', 'MAQ', 'RQ', 'TFQ', 'TFQ2', 'TFQ3']
PROMPT_ORDER = ['ao', 'cot', 'cove']

# 모든 merged 파일 불러오기
files = list(data_path.glob('*_merged.xlsx'))
print(f"총 {len(files)}개의 파일을 찾았습니다.\n")

# 파일 목록 출력
for f in sorted(files):
    print(f"  - {f.name}")


## 1. 데이터 로드 및 전처리


In [ ]:
def parse_filename(filename):
    """파일명에서 모델명과 문항유형 추출"""
    name = filename.replace('_merged.xlsx', '')
    parts = name.rsplit('_', 1)
    if len(parts) == 2:
        model_name, question_type = parts
        return model_name, question_type
    return None, None

def normalize_answer(ans):
    """답변 정규화 (대소문자 통일, 공백 제거, 정렬)"""
    if pd.isna(ans) or ans == 'none' or ans == '':
        return None
    
    # 문자열로 변환
    ans_str = str(ans).strip().upper()
    
    # 특수 케이스 처리
    if ans_str in ['NONE', 'NAN', '']:
        return None
    
    # 쉼표로 구분된 답변 정렬 (예: "B,A" -> "A,B")
    if ',' in ans_str:
        items = [item.strip() for item in ans_str.split(',')]
        items = sorted(set(items))  # 중복 제거 및 정렬
        return ','.join(items)
    
    return ans_str

def check_correct(label, answer):
    """정답 여부 확인"""
    label_norm = normalize_answer(label)
    answer_norm = normalize_answer(answer)
    
    # 둘 다 None이면 False
    if label_norm is None or answer_norm is None:
        return False
    
    return label_norm == answer_norm


In [ ]:
# 모든 데이터 로드 및 정확도 계산
results = []

for file_path in files:
    model_name, question_type = parse_filename(file_path.name)
    if not model_name or not question_type:
        continue
    
    df = pd.read_excel(file_path)
    
    # 각 프롬프트별 정확도 계산
    prompts = ['ao_ans', 'cot_ans', 'cove_ans']
    
    for prompt in prompts:
        if prompt in df.columns:
            # 정답 여부 체크
            correct = df.apply(lambda row: check_correct(row['label'], row[prompt]), axis=1)
            accuracy = correct.sum() / len(df) * 100
            
            # n_or_p 컬럼이 있으면 neg/pos별 정확도도 계산
            if 'n_or_p' in df.columns:
                neg_acc = correct[df['n_or_p'] == 'neg'].sum() / (df['n_or_p'] == 'neg').sum() * 100 if (df['n_or_p'] == 'neg').sum() > 0 else 0
                pos_acc = correct[df['n_or_p'] == 'pos'].sum() / (df['n_or_p'] == 'pos').sum() * 100 if (df['n_or_p'] == 'pos').sum() > 0 else 0
            else:
                neg_acc = None
                pos_acc = None
            
            results.append({
                'model': model_name,
                'question_type': question_type,
                'prompt': prompt.replace('_ans', ''),
                'accuracy': accuracy,
                'total_count': len(df),
                'correct_count': correct.sum(),
                'neg_accuracy': neg_acc,
                'pos_accuracy': pos_acc
            })

# 결과를 DataFrame으로 변환
results_df = pd.DataFrame(results)

# 모델과 문항 유형을 카테고리로 변환하여 순서 지정
results_df['model'] = pd.Categorical(results_df['model'], categories=MODEL_ORDER, ordered=True)
results_df['question_type'] = pd.Categorical(results_df['question_type'], categories=QTYPE_ORDER, ordered=True)
results_df['prompt'] = pd.Categorical(results_df['prompt'], categories=PROMPT_ORDER, ordered=True)

print(f"\\n총 {len(results_df)}개의 결과를 계산했습니다.")
print(f"\\n샘플 데이터:")
display(results_df.head(10))


In [ ]:
results_df.to_excel("results.xlsx")

## 2. 기본 정확도 분석


### 2.1 프롬프트별 전체 평균 정확도


In [ ]:
prompt_avg = results_df.groupby('prompt')['accuracy'].mean()
# 순서 지정
prompt_avg = prompt_avg.reindex(PROMPT_ORDER)

print("=" * 50)
print("프롬프트별 전체 평균 정확도")
print("=" * 50)
for prompt, acc in prompt_avg.items():
    print(f"{prompt:10s}: {acc:6.2f}%")
print("=" * 50)


### 2.2 모델별 전체 평균 정확도


In [ ]:
model_avg = results_df.groupby('model')['accuracy'].mean()
# 순서 지정
model_avg = model_avg.reindex(MODEL_ORDER)

print("=" * 50)
print("모델별 전체 평균 정확도")
print("=" * 50)
for model, acc in model_avg.items():
    print(f"{model:15s}: {acc:6.2f}%")
print("=" * 50)


### 2.3 문항유형별 전체 평균 정확도


In [ ]:
qtype_avg = results_df.groupby('question_type')['accuracy'].mean()
# 순서 지정
qtype_avg = qtype_avg.reindex(QTYPE_ORDER)

print("=" * 50)
print("문항유형별 전체 평균 정확도")
print("=" * 50)
for qtype, acc in qtype_avg.items():
    print(f"{qtype:5s}: {acc:6.2f}%")
print("=" * 50)


### 2.4 모델 × 프롬프트 상세 정확도


In [ ]:
model_prompt_pivot = results_df.pivot_table(
    index='model', 
    columns='prompt', 
    values='accuracy', 
    aggfunc='mean'
)
# 순서 지정
model_prompt_pivot = model_prompt_pivot.reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)

print("\\n모델 × 프롬프트 평균 정확도 (%):")
display(model_prompt_pivot.round(2))


### 2.5 문항유형 × 프롬프트 상세 정확도


In [ ]:
qtype_prompt_pivot = results_df.pivot_table(
    index='question_type', 
    columns='prompt', 
    values='accuracy', 
    aggfunc='mean'
)
# 순서 지정
qtype_prompt_pivot = qtype_prompt_pivot.reindex(index=QTYPE_ORDER, columns=PROMPT_ORDER)

print("\\n문항유형 × 프롬프트 평균 정확도 (%):")
display(qtype_prompt_pivot.round(2))


## 3. 심화 분석


### 3.1 프롬프트 효과 분석 (개선률)


In [ ]:
# 각 모델-문항유형 조합별로 프롬프트 효과 계산
improvement_results = []

for (model, qtype), group in results_df.groupby(['model', 'question_type']):
    # 각 프롬프트의 정확도 가져오기
    ao_acc = group[group['prompt'] == 'ao']['accuracy'].values
    cot_acc = group[group['prompt'] == 'cot']['accuracy'].values
    cove_acc = group[group['prompt'] == 'cove']['accuracy'].values
    
    if len(ao_acc) > 0 and len(cot_acc) > 0 and len(cove_acc) > 0:
        ao_acc = ao_acc[0]
        cot_acc = cot_acc[0]
        cove_acc = cove_acc[0]
        
        # 개선률 계산 (절대값)
        ao_to_cot = cot_acc - ao_acc
        cot_to_cove = cove_acc - cot_acc
        ao_to_cove = cove_acc - ao_acc
        
        # 개선률 계산 (상대값, %)
        ao_to_cot_pct = (cot_acc - ao_acc) / ao_acc * 100 if ao_acc > 0 else 0
        cot_to_cove_pct = (cove_acc - cot_acc) / cot_acc * 100 if cot_acc > 0 else 0
        ao_to_cove_pct = (cove_acc - ao_acc) / ao_acc * 100 if ao_acc > 0 else 0
        
        improvement_results.append({
            'model': model,
            'question_type': qtype,
            'ao_acc': ao_acc,
            'cot_acc': cot_acc,
            'cove_acc': cove_acc,
            'ao→cot (절대)': ao_to_cot,
            'cot→cove (절대)': cot_to_cove,
            'ao→cove (절대)': ao_to_cove,
            'ao→cot (%)': ao_to_cot_pct,
            'cot→cove (%)': cot_to_cove_pct,
            'ao→cove (%)': ao_to_cove_pct
        })

improvement_df = pd.DataFrame(improvement_results)

print("\\n프롬프트 개선 효과 (절대값 - %p):")
display(improvement_df[['model', 'question_type', 'ao→cot (절대)', 'cot→cove (절대)', 'ao→cove (절대)']].round(2))

print("\\n프롬프트 개선 효과 (상대값 - %):")
display(improvement_df[['model', 'question_type', 'ao→cot (%)', 'cot→cove (%)', 'ao→cove (%)']].round(2))


In [ ]:
# 모델별 평균 개선률
model_improvement = improvement_df.groupby('model')[['ao→cot (절대)', 'cot→cove (절대)', 'ao→cove (절대)']].mean()

print("\\n모델별 평균 개선률 (%p):")
display(model_improvement.round(2))

# 문항유형별 평균 개선률
qtype_improvement = improvement_df.groupby('question_type')[['ao→cot (절대)', 'cot→cove (절대)', 'ao→cove (절대)']].mean()

print("\\n문항유형별 평균 개선률 (%p):")
display(qtype_improvement.round(2))


### 3.2 문항 난이도 분석 (문항유형별)


In [ ]:
# 문항유형별 평균 정확도와 표준편차
qtype_difficulty = results_df.groupby('question_type').agg({
    'accuracy': ['mean', 'std', 'min', 'max']
}).round(2)

print("\\n문항유형별 난이도 분석:")
print("(정확도가 낮을수록 어려운 문항)")
display(qtype_difficulty)

# 난이도 순위
qtype_ranked = results_df.groupby('question_type')['accuracy'].mean().sort_values()
print("\\n난이도 순위 (어려운 순):")
for i, (qtype, acc) in enumerate(qtype_ranked.items(), 1):
    print(f"{i}. {qtype}: {acc:.2f}%")


### 3.3 Negative vs Positive 질문 분석


In [ ]:
# neg/pos 정확도가 있는 데이터만 필터링
neg_pos_data = results_df[results_df['neg_accuracy'].notna()].copy()

if len(neg_pos_data) > 0:
    # 전체 평균
    overall_neg = neg_pos_data['neg_accuracy'].mean()
    overall_pos = neg_pos_data['pos_accuracy'].mean()
    
    print("\\nNegative vs Positive 질문 정확도:")
    print("=" * 50)
    print(f"Negative 질문 평균: {overall_neg:.2f}%")
    print(f"Positive 질문 평균: {overall_pos:.2f}%")
    print(f"차이: {overall_neg - overall_pos:.2f}%p")
    print("=" * 50)
    
    # 모델별 neg/pos 정확도
    model_neg_pos = neg_pos_data.groupby('model')[['neg_accuracy', 'pos_accuracy']].mean()
    print("\\n모델별 Negative vs Positive:")
    display(model_neg_pos.round(2))
else:
    print("\\nNegative/Positive 구분 데이터가 없습니다.")


## 4. 시각화


### 4.1 히트맵: 모델 × 프롬프트 정확도


In [ ]:
# plots 폴더 생성
plots_path = Path('plots')
plots_path.mkdir(exist_ok=True)

# 문항유형별로 히트맵 6개 생성 (통합)
fig, axes = plt.subplots(2, 3, figsize=(24, 12))
axes = axes.flatten()

for idx, qtype in enumerate(QTYPE_ORDER):
    qtype_data = results_df[results_df['question_type'] == qtype]
    
    # 피벗 테이블 생성
    pivot = qtype_data.pivot_table(
        index='model', 
        columns='prompt', 
        values='accuracy'
    )
    # 순서 지정
    pivot = pivot.reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)
    
    # 히트맵 그리기
    sns.heatmap(
        pivot, 
        annot=True, 
        fmt='.1f', 
        cmap='RdYlBu_r',  # 빨강(낮음) -> 노랑 -> 파랑(높음)
        cbar_kws={'label': '정확도 (%)'},
        vmin=0,
        vmax=100,
        ax=axes[idx],
        linewidths=0.5
    )
    
    axes[idx].set_title(f'{qtype} - 모델, 프롬프트별 정확도', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('프롬프트', fontsize=12)
    axes[idx].set_ylabel('모델', fontsize=12)

plt.tight_layout()
plt.savefig('heatmap_model_prompt_by_qtype.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n통합 히트맵이 'heatmap_model_prompt_by_qtype.png'로 저장되었습니다.")

# 개별 히트맵 저장
for qtype in QTYPE_ORDER:
    qtype_data = results_df[results_df['question_type'] == qtype]
    
    # 피벗 테이블 생성
    pivot = qtype_data.pivot_table(
        index='model', 
        columns='prompt', 
        values='accuracy'
    )
    # 순서 지정
    pivot = pivot.reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)
    
    # 개별 그래프 생성
    fig, ax = plt.subplots(figsize=(8, 6))
    
    sns.heatmap(
        pivot, 
        annot=True, 
        fmt='.1f', 
        cmap='RdYlBu_r',
        cbar_kws={'label': '정확도 (%)'},
        vmin=0,
        vmax=100,
        ax=ax,
        linewidths=0.5
    )
    
    ax.set_title(f'{qtype} - 모델, 프롬프트별 정확도', fontsize=14, fontweight='bold')
    ax.set_xlabel('프롬프트', fontsize=12)
    ax.set_ylabel('모델', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(plots_path / f'heatmap_{qtype}.png', dpi=300, bbox_inches='tight')
    plt.close()

print(f"개별 히트맵들이 '{plots_path}' 폴더에 저장되었습니다.")


In [ ]:
# 전체 평균 히트맵 (문항유형 통합)
fig, ax = plt.subplots(figsize=(10, 6))

overall_pivot = results_df.pivot_table(
    index='model', 
    columns='prompt', 
    values='accuracy',
    aggfunc='mean'
)
# 순서 지정
overall_pivot = overall_pivot.reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)

sns.heatmap(
    overall_pivot, 
    annot=True, 
    fmt='.1f', 
    cmap='RdYlBu_r',  # 빨강(낮음) -> 노랑 -> 파랑(높음)
    cbar_kws={'label': '정확도 (%)'},
    vmin=0,
    vmax=100,
    ax=ax,
    linewidths=1,
    linecolor='white'
)

ax.set_title('프롬프트별 정확도 평균', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('프롬프트', fontsize=14)
ax.set_ylabel('모델', fontsize=14)

plt.tight_layout()
plt.savefig('heatmap_overall_average.png', dpi=300, bbox_inches='tight')
plt.show()

print("\\n전체 평균 히트맵이 'heatmap_overall_average.png'로 저장되었습니다.")


### 4.2 그룹 막대 그래프: 모델별 프롬프트 비교


In [ ]:
# 문항유형별로 서브플롯 생성
fig, axes = plt.subplots(2, 3, figsize=(27, 12))
axes = axes.flatten()

# 색상 팔레트 설정
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']

for idx, qtype in enumerate(QTYPE_ORDER):
    qtype_data = results_df[results_df['question_type'] == qtype]
    
    # 피벗 테이블
    pivot = qtype_data.pivot_table(
        index='model', 
        columns='prompt', 
        values='accuracy'
    )
    # 순서 지정
    pivot = pivot.reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)
    
    # 막대 그래프
    pivot.plot(
        kind='bar',
        ax=axes[idx],
        color=colors,
        width=0.8,
        edgecolor='black',
        linewidth=0.5
    )
    
    axes[idx].set_title(f'{qtype} 문항', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('모델', fontsize=15)
    axes[idx].set_ylabel('정확도 (%)', fontsize=13)
    axes[idx].set_ylim(0, 100)
    axes[idx].legend(title='프롬프트', loc='upper right')
    axes[idx].grid(axis='y', alpha=0.3)
    axes[idx].set_xticklabels(pivot.index, rotation=45, ha='right')
    
    # 값 표시
    for container in axes[idx].containers:
        axes[idx].bar_label(container, fmt='%.1f', fontsize=8)

plt.suptitle('문항 유형, 모델별 정확도', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('grouped_bar_by_qtype.png', dpi=600, bbox_inches='tight')
plt.show()

print("\\n통합 그룹 막대 그래프가 'grouped_bar_by_qtype.png'로 저장되었습니다.")

# 개별 막대 그래프 저장
for qtype in QTYPE_ORDER:
    qtype_data = results_df[results_df['question_type'] == qtype]
    
    # 피벗 테이블
    pivot = qtype_data.pivot_table(
        index='model', 
        columns='prompt', 
        values='accuracy'
    )
    # 순서 지정
    pivot = pivot.reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)
    
    # 개별 그래프 생성
    fig, ax = plt.subplots(figsize=(10, 7))
    
    pivot.plot(
        kind='bar',
        ax=ax,
        color=colors,
        width=0.7,
        edgecolor='black',
        linewidth=0.5
    )
    
    ax.set_title(f'{qtype} 문항 - 모델별 프롬프트 정확도', fontsize=14, fontweight='bold')
    ax.set_xlabel('모델', fontsize=12)
    ax.set_ylabel('정확도 (%)', fontsize=12)
    ax.set_ylim(0, 100)
    ax.legend(title='프롬프트', loc='upper right')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(pivot.index, rotation=45, ha='right')
    
    # 값 표시
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(plots_path / f'grouped_bar_{qtype}.png', dpi=300, bbox_inches='tight')
    plt.close()

print(f"개별 그룹 막대 그래프들이 '{plots_path}' 폴더에 저장되었습니다.")


In [ ]:
# 전체 평균 그룹 막대 그래프
fig, ax = plt.subplots(figsize=(12, 7))

overall_pivot = results_df.pivot_table(
    index='model', 
    columns='prompt', 
    values='accuracy',
    aggfunc='mean'
)
# 순서 지정
overall_pivot = overall_pivot.reindex(index=MODEL_ORDER, columns=PROMPT_ORDER)

overall_pivot.plot(
    kind='bar',
    ax=ax,
    color=colors,
    width=0.7,
    edgecolor='black',
    linewidth=1
)

ax.set_title('모델, 프롬프트별 정확도 평균 비교', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('모델', fontsize=14)
ax.set_ylabel('정확도 (%)', fontsize=14)
ax.set_ylim(0, 100)
ax.legend(title='프롬프트', fontsize=12, title_fontsize=12, loc='upper right')
ax.grid(axis='y', alpha=0.3)
ax.set_xticklabels(overall_pivot.index, rotation=45, ha='right', fontsize=12)

# 값 표시
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=10)

plt.tight_layout()
plt.savefig('grouped_bar_overall.png', dpi=300, bbox_inches='tight')
plt.show()

print("\\n전체 평균 그룹 막대 그래프가 'grouped_bar_overall.png'로 저장되었습니다.")


## 5. 결과 요약

이 분석을 통해 다음을 확인할 수 있습니다:

### 📊 주요 발견사항
1. **프롬프트별 성능**: ao → cot → cove 순서로 성능이 개선되는지 확인
2. **모델별 성능**: 어떤 모델이 가장 높은 정확도를 보이는지
3. **문항유형별 난이도**: 어떤 문항 유형이 가장 어려운지
4. **프롬프트 효과**: 각 프롬프트 전략이 얼마나 효과적인지
5. **Negative vs Positive**: 부정 질문과 긍정 질문 중 어느 것이 더 어려운지

### 📁 생성된 파일
- `heatmap_model_prompt_by_qtype.png`: 문항유형별 히트맵 (2×2)
- `heatmap_overall_average.png`: 전체 평균 히트맵
- `grouped_bar_by_qtype.png`: 문항유형별 그룹 막대 그래프 (2×2)
- `grouped_bar_overall.png`: 전체 평균 그룹 막대 그래프
- `line_chart_by_qtype.png`: 문항유형별 선 그래프 (2×2)
- `line_chart_overall.png`: 전체 평균 선 그래프

모든 그래프는 고해상도(300 dpi)로 저장되어 논문이나 발표 자료에 바로 사용할 수 있습니다.
